In this notebook we train two models (model_start and model_end) and then a third model_theta.

In [3]:
from ..src.models import MyNet, Lenet5, tiny, MyNet_small, CIFAR10ConvNet
import torch
from scheduler import make_scheduler
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

root = "."
datafolder = f"{root}/data"
base_directory = f"{root}/results_notebook"

ImportError: attempted relative import with no known parent package

In [ ]:
retrain=True

In [ ]:
MODEL = CIFAR10ConvNet
model_kargs = {"dropout": 0.5}
loss_fn = torch.nn.CrossEntropyLoss(weight=None, size_average=None, ignore_index=-100, reduce=None, reduction='mean', label_smoothing=0.0)
batch_size = 256

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.CIFAR10(root=f'{datafolder}/train', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root=f'{datafolder}/test', train=False, download=True, transform=transform)
subset_test = Subset(test_dataset, indices=range(len(test_dataset) // 1))
subset_train = Subset(train_dataset, indices=range(len(train_dataset) // 1))
test_loader = DataLoader(subset_test, batch_size=batch_size)
train_loader = DataLoader(subset_train, batch_size=batch_size)


In [ ]:
if retrain:
    model_lr_start = 1e-4
    model_lr_end = 2e-3
    model_epochs = 100

    total_iter = model_epochs*train_loader.__len__()

    model_start = MODEL(**model_kargs).to(device)
    optimizer_start = torch.optim.Adam(params=model_start.parameters(), lr=model_lr_end.clone())
    scheduler_start = make_scheduler(optimizer_start, 
                                             train_num_steps=total_iter, 
                                             lr_start_warmup=model_lr_start.clone(), 
                                             lr=model_lr_end.clone(), 
                                             lr_warmup_steps=5*train_loader.__len__(), 
                                             lr_finetune_halftime=total_iter // (5*3), 
                                             lr_finetune_steps=total_iter // 3
            )
    model_start, all_train_losses, lrs, epoch_train_losses, test_losses, epoch_train_accuracy, plots = train(model_start, 
                                                train_loader=train_loader, 
                                                test_loader=test_loader, 
                                                optimizer=optimizer_start, 
                                                scheduler=scheduler_start, 
                                                epochs=model_epochs, loss_fn=loss_fn, device=device, 
                                                plot=True, plotpath=f"{base_directory}/start_model/figures",
                                                verbose=True
                                                )